# Giai đoạn 3: Đặc trưng Vận hành Liên Doanh nghiệp (Hồi 3 — Bằng chứng đối chiếu)

## 00. Giới thiệu giai đoạn

- **Tên giai đoạn:** Giai đoạn 3 — Đặc trưng vận hành liên doanh nghiệp (`03_cross_business_eda.ipynb`).
- **Vai trò trong câu chuyện dữ liệu (Hồi 3 — *Bằng chứng đối chiếu*):** So sánh nhịp hoạt động, cơ cấu vận hành và biến động thời gian giữa các nhóm doanh nghiệp **chỉ trên những chiều thực sự tương đồng về ngữ nghĩa, đơn vị và khung thời gian**, qua đó làm rõ điểm chung và nét đặc thù của từng ngành nghề.
- **Mục tiêu:**
  1. Xây dựng Ma trận khả năng so sánh (cấp độ chi tiết `granularity`, ngữ nghĩa trường dữ liệu, có giờ giao dịch hay không, khoảng thời gian giao nhau).
  2. Chuẩn hóa chỉ số về **tỉ lệ / phần trăm / chỉ số index** (Chương 2 §2.6) để loại bỏ nhiễu do chênh lệch quy mô.
  3. Phân tích nhịp vận hành (giờ/ca, ngày thường vs. cuối tuần) dạng *Small Multiples* và biến động dịp Tết qua `calendar.csv`.
  4. Tổng hợp đầy đủ các giới hạn đối chiếu và trường hợp không thể so sánh trực tiếp.
- **Phạm vi dữ liệu:** Chỉ đọc `public_test` (`dev_001`–`dev_014`, `calendar.csv`, `rulebook.md`).
- **Đầu ra mong đợi (Cột mốc M3 phần Hồi 3) & Nơi lưu trữ:**
  - Thư mục gốc phase: `artifacts/guided_eda/phase3_cross_business/`
  - Bảng kết quả (`PHASE3_TABLES_DIR`): `tables/comparability_matrix.csv`, `tables/normalized_cross_metrics.csv`, `tables/operating_rhythm_summary.csv`, `tables/tet_window_comparison.csv`, `tables/cross_business_limitations.csv`.
  - Biểu đồ Hồi 3 (`PHASE3_FIGURES_DIR`): `figures/` (Small Multiples nhịp vận hành và biểu đồ biến động dịp Tết).
- **Các giới hạn cần tuân thủ:**
  - **Không** coi hai cột cùng tên ở hai doanh nghiệp là tương đương về ngữ nghĩa nếu chưa kiểm chứng.
  - **Không** so sánh trực tiếp các chỉ số khác bản chất (ví dụ: cấp hóa đơn với cấp dòng hàng) hoặc khác cửa sổ thời gian (cutoff không đồng đều).
  - **Không** suy diễn kết quả xếp hạng "doanh nghiệp tốt nhất" hay quy luật mùa vụ dài hạn từ dữ liệu đứt quãng.


## 01. Thiết lập môi trường


In [ ]:
import sys
from pathlib import Path

# Tự động xác định thư mục gốc dự án và thêm src vào sys.path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "common").is_dir():
    for parent in PROJECT_ROOT.parents:
        if (parent / "src" / "common").is_dir():
            PROJECT_ROOT = parent
            break

SCRIPT_ROOT = PROJECT_ROOT / "src"
if not SCRIPT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục mã nguồn EDA tại: {SCRIPT_ROOT}"
    )

if str(SCRIPT_ROOT) not in sys.path:
    sys.path.insert(0, str(SCRIPT_ROOT))

print(f"Thư mục gốc dự án : {PROJECT_ROOT}")
print(f"Thư mục script EDA : {SCRIPT_ROOT}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import có chọn lọc các tiện ích từ src/
from common.config import load_config
from common.paths import (
    DatasetContext,
    assert_not_in_raw,
    assert_safe_output,
    is_benchmark_file,
)
from readers.csv_reader import read_all_csv_rows
from readers.excel_reader import list_sheets, read_sheet_rows
from readers.sqlite_reader import inspect_sqlite, query_readonly
from temporal.calendar_registry import validate_calendar
from temporal.coverage import (
    daily_counts_from_dates,
    monthly_counts_from_dates,
    weekday_counts_from_dates,
)
from cross_business.schema_compare import ROLE_TOKENS, detect_field_role

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

# Thiết lập giao diện biểu đồ cơ bản (hỗ trợ hiển thị inline trong Jupyter)
# Lưu ý: Không import visualization.figure_theme vì module đó ép backend Agg.
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titlelocation": "left",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": ":",
    "grid.alpha": 0.55,
    "axes.unicode_minus": False,
    "figure.figsize": (10, 5.5),
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

# Bảng màu gợi ý theo nguyên tắc "Bắt đầu từ màu xám" (Schwabish)
COLOR_GRAY = "#9AA5B1"
COLOR_GRAY_DARK = "#52606D"
COLOR_ACCENT_BLUE = "#1769AA"
COLOR_ACCENT_TEAL = "#087B5A"
COLOR_ACCENT_AMBER = "#A36516"
COLOR_ACCENT_RED = "#B32336"


## 02. Đọc dữ liệu, Khởi tạo thư mục đầu ra & Hợp đồng đầu vào

Notebook này chạy độc lập với Notebook 01 và 02. Phần dưới đây nạp sẵn nguồn `public_test`, lịch âm/dương `calendar.csv`, khởi tạo (`mkdir`) các thư mục đầu ra của Giai đoạn 3 và thiết lập **Hợp đồng đầu vào (Input Contract)** từ Giai đoạn 1 và 2.


In [ ]:
config = load_config(workspace_root=PROJECT_ROOT)
context: DatasetContext = config.context_for("public_test")
assert context.read_only is True
assert not is_benchmark_file(context.calendar_path)

business_ids = context.business_ids()
calendar_registry = validate_calendar(context.calendar_path, split="public_test")
_cal_header, cal_rows = read_all_csv_rows(context.calendar_path)
calendar_df = pd.DataFrame(cal_rows)
calendar_df["date"] = pd.to_datetime(calendar_df["date"])

# Khởi tạo (mkdir) thư mục lưu trữ đầu ra cho Giai đoạn 3 (nằm ngoài data/ và raw_data/)
GUIDED_ROOT = config.output_root / "guided_eda"
PHASE3_ROOT = GUIDED_ROOT / "phase3_cross_business"
PHASE3_TABLES_DIR = PHASE3_ROOT / "tables"
PHASE3_FIGURES_DIR = PHASE3_ROOT / "figures"

assert_not_in_raw(config.raw_root, GUIDED_ROOT)
for target_dir in (PHASE3_ROOT, PHASE3_TABLES_DIR, PHASE3_FIGURES_DIR):
    assert_safe_output(config.output_root, target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

print(f"Đã kết nối public_test : {context.root} ({len(business_ids)} doanh nghiệp)")
print(f"Đã nạp calendar.csv    : {len(calendar_df)} ngày ({calendar_registry.min_date} -> {calendar_registry.max_date})")
print(f"Thư mục bảng Phase 3   : {PHASE3_TABLES_DIR}")
print(f"Thư mục hình Phase 3   : {PHASE3_FIGURES_DIR}")


In [ ]:
# ==============================================================================
# HỢP ĐỒNG ĐẦU VÀO TỪ GIAI ĐOẠN 1 & 2 (NB01 & NB02 -> NB03)
# ==============================================================================
DEFAULT_PHASE1_CATALOG_PATH = GUIDED_ROOT / "phase1_dataset_discovery" / "tables" / "catalog_14.csv"
DEFAULT_PHASE2_CLEAN_DIR = GUIDED_ROOT / "phase2_business_eda" / "tidy_data"

PHASE1_CATALOG_PATH: Path | None = DEFAULT_PHASE1_CATALOG_PATH
PHASE2_CLEAN_DATA_DIR: Path | None = DEFAULT_PHASE2_CLEAN_DIR

# Hoặc nạp/khai báo trực tiếp các bảng Tidy và nhóm doanh nghiệp có thể so sánh
PHASE1_CATALOG_DF: pd.DataFrame | None = None
PHASE2_BUSINESS_TABLES: dict[str, pd.DataFrame] = {}
COMPARABLE_BUSINESS_GROUPS: dict[str, list[str]] = {}


def check_phase3_input_contract(
    catalog_path: Path | None = PHASE1_CATALOG_PATH,
    clean_dir: Path | None = PHASE2_CLEAN_DATA_DIR,
    business_tables: dict[str, pd.DataFrame] = PHASE2_BUSINESS_TABLES,
    comparable_groups: dict[str, list[str]] = COMPARABLE_BUSINESS_GROUPS,
) -> tuple[pd.DataFrame | None, dict[str, pd.DataFrame], dict[str, str]]:
    status_report: dict[str, str] = {}
    catalog_df = PHASE1_CATALOG_DF
    tables = dict(business_tables)

    if catalog_path is not None:
        p = Path(catalog_path)
        if not p.is_file():
            if p == DEFAULT_PHASE1_CATALOG_PATH:
                status_report["phase1_catalog"] = (
                    f"[CHỜ BỔ SUNG] Chưa có tệp bàn giao từ Giai đoạn 1 tại: {p}"
                )
            else:
                status_report["phase1_catalog"] = f"[CẢNH BÁO] Không tìm thấy tệp: {p}"
        elif p.suffix.lower() == ".csv":
            catalog_df = pd.read_csv(p)
            status_report["phase1_catalog"] = f"[ĐÃ SẴN SÀNG] Nạp từ tệp ({len(catalog_df)} dòng): {p.name}"
        elif p.suffix.lower() == ".json":
            catalog_df = pd.read_json(p)
            status_report["phase1_catalog"] = f"[ĐÃ SẴN SÀNG] Nạp từ tệp ({len(catalog_df)} dòng): {p.name}"
        else:
            status_report["phase1_catalog"] = f"[CẢNH BÁO] Định dạng chưa hỗ trợ: {p.suffix}"
    elif catalog_df is not None:
        status_report["phase1_catalog"] = f"[ĐÃ SẴN SÀNG] Nhập trực tiếp ({len(catalog_df)} dòng)"
    else:
        status_report["phase1_catalog"] = "[CHỜ BỔ SUNG] Chưa có danh mục 14 doanh nghiệp từ Giai đoạn 1 (Notebook 01)"

    if clean_dir is not None:
        d = Path(clean_dir)
        if not d.is_dir():
            if d == DEFAULT_PHASE2_CLEAN_DIR:
                status_report["phase2_clean_data"] = (
                    f"[CHỜ BỔ SUNG] Chưa có thư mục dữ liệu Tidy từ Giai đoạn 2 tại: {d}"
                )
            else:
                status_report["phase2_clean_data"] = f"[CẢNH BÁO] Không tìm thấy thư mục: {d}"
        else:
            loaded_files = sorted(d.glob("*.csv"))
            for fp in loaded_files:
                tables[fp.stem] = pd.read_csv(fp)
            if loaded_files:
                status_report["phase2_clean_data"] = (
                    f"[ĐÃ SẴN SÀNG] Đã nạp {len(loaded_files)} tệp CSV sạch từ {d.name}"
                )
            else:
                status_report["phase2_clean_data"] = (
                    f"[CHỜ BỔ SUNG] Thư mục {d.name} hiện đang trống (chưa có tệp *_tidy.csv từ Notebook 02)"
                )
    elif tables:
        status_report["phase2_clean_data"] = f"[ĐÃ SẴN SÀNG] Nhập trực tiếp ({len(tables)} bảng)"
    else:
        status_report["phase2_clean_data"] = "[CHỜ BỔ SUNG] Chưa có bảng dữ liệu sạch từ Giai đoạn 2 (Notebook 02)"

    if comparable_groups:
        status_report["comparable_groups"] = (
            f"[ĐÃ SẴN SÀNG] Đã khai báo {len(comparable_groups)} nhóm đối chiếu"
        )
    else:
        status_report["comparable_groups"] = "[CHỜ BỔ SUNG] Chờ xây dựng tại T3.1a"

    for key, msg in status_report.items():
        print(f"[Hợp đồng đầu vào - {key}]: {msg}")
    return catalog_df, tables, status_report


phase1_catalog_df, phase2_tables, phase3_contract_status = check_phase3_input_contract()


## 03. Tổng quan dữ liệu cơ bản


In [ ]:
# Xem nhanh các cửa sổ Tết âm lịch trích từ calendar.csv (23 tháng Chạp -> mùng 10 tháng Giêng)
print("Các cửa sổ Tết theo rulebook (23 tháng Chạp -> 10 tháng Giêng):")
for start_d, end_d in calendar_registry.tet_windows:
    print(f"  - {start_d} đến {end_d}")

print("\nCác vai trò ngữ nghĩa gợi ý (ROLE_TOKENS) dùng hỗ trợ rà soát cột:")
print(list(ROLE_TOKENS.keys()))

display(calendar_df.head(5))


## 04. Các tác vụ khám phá dữ liệu (EDA Tasks)


## Tác vụ T3.1a — Lập ma trận khả năng so sánh liên doanh nghiệp

### Mục tiêu
Đánh giá có hệ thống những doanh nghiệp nào có thể đối chiếu với nhau trên từng chiều phân tích (cấp độ chi tiết, có mốc giờ giao dịch, khoảng thời gian hoạt động giao nhau, đơn vị tính) và loại trừ các phép so sánh không hợp lệ.

### Câu hỏi khám phá
- Những nhóm doanh nghiệp nào có cùng cấp độ chi tiết (`granularity`: cấp đơn vs cấp dòng hàng)?
- Những doanh nghiệp nào có dữ liệu giờ giao dịch (`HH:MM`) để so sánh nhịp theo giờ hoặc theo ca sáng/chiều?
- Khoảng thời gian hoạt động giao nhau chung (intersection window) giữa các doanh nghiệp cần so sánh là từ ngày nào đến ngày nào?

### Các bước thực hiện

**Bước 1 — Tổng hợp đặc điểm dữ liệu từ Giai đoạn 1 và 2**
Sử dụng `phase1_catalog_df` / `phase2_tables` (hoặc tự khảo sát cột chính của các doanh nghiệp quan tâm từ `context`) để xem các trường hiện có.

**Bước 2 — Kiểm tra tính tương đương ngữ nghĩa thực tế của các trường**
Có thể gợi ý vai trò bằng `detect_field_role(col_name)` nhưng **bắt buộc kiểm chứng** định nghĩa thực tế trong `README.md` và `rulebook.md`.

**Bước 3 — Kiểm tra sự hiện diện của mốc giờ giao dịch & ngày đặt hàng**
Phân biệt rạch ròi doanh nghiệp chỉ có ngày (`YYYY-MM-DD`) với doanh nghiệp có đầy đủ giờ phút (`HH:MM`) hoặc cột ca làm việc.

**Bước 4 — Xác định cửa sổ thời gian giao nhau hợp lệ**
Tính giao của `[min_date, max_date]` cho từng nhóm doanh nghiệp dự kiến đối chiếu và đánh dấu các tháng bị cắt cụt (censored) hoặc bị trống dữ liệu.

**Bước 5 — Lập Ma trận khả năng so sánh & xuất `comparability_matrix.csv`**
Tạo `comparability_matrix_df`, cập nhật `COMPARABLE_BUSINESS_GROUPS` và lưu vào `PHASE3_TABLES_DIR / 'comparability_matrix.csv'`.

### Gợi ý phương pháp & Lưu ý
- Tuyệt đối không coi hai cột có cùng tên là tương đương về ngữ nghĩa (ví dụ: một dòng trong tiệm sửa xe là 1 lượt xe, trong khi một dòng ở tạp hóa là 1 mặt hàng trong đơn).
- Chỉ gộp nhóm so sánh khi đã xác nhận cùng `grain`, cùng đơn vị/loại tiền và cùng quy tắc lọc đơn hủy/trả theo `rulebook.md`.

### Triển khai


In [ ]:
# TODO (T3.1a — Bước 1: Tổng hợp đặc điểm dữ liệu từ Giai đoạn 1 và 2)
# TODO: Tổng hợp thông tin cấp độ chi tiết, cột ngày/giờ, cột số tiền và min_date..max_date cho các doanh nghiệp


In [ ]:
# TODO (T3.1a — Bước 2: Kiểm tra tính tương đương ngữ nghĩa thực tế của các trường)
# TODO: Đối chiếu ngữ nghĩa thực tế giữa các cột cùng vai trò (không tin mù quáng tên cột giống nhau)


In [ ]:
# TODO (T3.1a — Bước 3: Kiểm tra sự hiện diện của mốc giờ giao dịch & ngày đặt hàng)
# TODO: Phân loại doanh nghiệp theo khả năng phân tích giờ/ca vs theo ngày/thứ


In [ ]:
# TODO (T3.1a — Bước 4: Xác định cửa sổ thời gian giao nhau hợp lệ)
# TODO: Tính cửa sổ thời gian giao nhau cho từng chiều phân tích (nhịp ngày/tuần, dịp Tết...)


In [ ]:
# TODO (T3.1a — Bước 5: Lập Ma trận khả năng so sánh & xuất `comparability_matrix.csv`)
# TODO: Xây dựng comparability_matrix_df, cập nhật COMPARABLE_BUSINESS_GROUPS và xuất CSV vào PHASE3_TABLES_DIR
comparability_matrix_df = pd.DataFrame()


### Trực quan hóa


In [ ]:
# TODO (T3.1a — Trực quan hóa)
# Vẽ heatmap/bảng ma trận khả năng so sánh (`comparability_matrix_df`) hoặc sơ đồ thanh ngang thể hiện cửa sổ thời gian giao nhau của các doanh nghiệp đủ điều kiện.
# Lưu ý: Không sử dụng biểu đồ 3D hoặc hai trục tung (dual axis); biểu đồ cột bắt đầu từ 0.


### Kết quả & Nhận định

> **TODO (T3.1a):** Thành viên tự ghi nhận phát hiện dựa trên bằng chứng vừa chạy ở trên (không suy diễn vượt quá dữ liệu).

- **Các nhóm doanh nghiệp đủ điều kiện đối chiếu cho từng chiều phân tích:** ...
- **Các trường hợp bị loại trừ khỏi phép so sánh trực tiếp và lý do:** ...


## Tác vụ T3.1b — Tính các chỉ số chuẩn hóa theo tỉ lệ / phần trăm

### Mục tiêu
Chuẩn hóa các đại lượng có quy mô khác nhau về tỉ trọng (%), tỉ số hoặc chỉ số tương đối (Chương 2 §2.6) trên cửa sổ thời gian hợp lệ để so sánh công bằng giữa các doanh nghiệp.

### Câu hỏi khám phá
- Vì sao không thể so sánh trực tiếp tổng doanh thu hoặc tổng số đơn thô giữa các doanh nghiệp khác quy mô và khác số ngày hoạt động?
- Cách chuẩn hóa nào (tỉ trọng % theo giờ/thứ, trung bình mỗi ngày có hoạt động, chỉ số index lấy ngày thường = 100%) phản ánh trung thực nhất bản chất từng chỉ số?
- Tổng các tỉ trọng thành phần sau khi chuẩn hóa có bảo toàn bằng 100% ở từng doanh nghiệp hay không?

### Các bước thực hiện

**Bước 1 — Xác định các chỉ số cần chuẩn hóa theo từng nhóm tương thích**
Chỉ chọn các chỉ số đã qua cổng kiểm tra ở `T3.1a` (ví dụ: tỷ trọng đơn/doanh thu theo khung giờ, tỷ lệ ngày thường vs cuối tuần, tỷ lệ hủy đơn).

**Bước 2 — Thiết kế công thức chuẩn hóa tỉ trọng / trung bình trên ngày lịch**
Đảm bảo mẫu số công bằng: ví dụ theo `rulebook.md`, *trung bình mỗi ngày thứ X trong kỳ* phải chia cho số ngày thứ X trong kỳ có độ phủ.

**Bước 3 — Thực thi tính toán, kiểm tra tổng tỉ trọng & xuất `normalized_cross_metrics.csv`**
Tính `normalized_cross_df` trên đúng cửa sổ thời gian giao nhau, kiểm tra `sum(share) == 100%` và lưu vào `PHASE3_TABLES_DIR / 'normalized_cross_metrics.csv'`.

**Bước 4 — Đánh giá khả năng diễn giải và độ nhạy mẫu số**
Kiểm tra xem doanh nghiệp có số lượng đơn quá ít hay bị lệch bởi vài ngày đặc biệt có làm méo tỉ lệ phần trăm hay không.

### Gợi ý phương pháp & Lưu ý
- Luôn giữ cột cỡ mẫu gốc `n` (số đơn, số ngày quan sát) song song với cột tỉ lệ % để hiển thị minh bạch trên biểu đồ.
- Nếu so sánh theo tháng, tránh đưa tháng bị cắt cụt ngày cutoff vào tính tỷ trọng trừ khi đã chuẩn hóa theo *trung bình/ngày* trên số ngày thực tế có dữ liệu.

### Triển khai


In [ ]:
# TODO (T3.1b — Bước 1: Xác định các chỉ số cần chuẩn hóa theo từng nhóm tương thích)
# TODO: Xác định danh sách chỉ số đối chiếu hợp lệ và mẫu số chuẩn hóa cho từng chỉ số


In [ ]:
# TODO (T3.1b — Bước 2: Thiết kế công thức chuẩn hóa tỉ trọng / trung bình trên ngày lịch)
# TODO: Viết các hàm chuẩn hóa (tỉ trọng %, trung bình theo ngày hoạt động, index so với baseline)


In [ ]:
# TODO (T3.1b — Bước 3: Thực thi tính toán, kiểm tra tổng tỉ trọng & xuất `normalized_cross_metrics.csv`)
# TODO: Tính normalized_cross_df, kiểm tra nhất quán mẫu số và lưu ra PHASE3_TABLES_DIR
normalized_cross_df = pd.DataFrame()


In [ ]:
# TODO (T3.1b — Bước 4: Đánh giá khả năng diễn giải và độ nhạy mẫu số)
# TODO: Kiểm tra cỡ mẫu n của từng doanh nghiệp sau khi lọc cửa sổ thời gian chung


### Trực quan hóa


In [ ]:
# TODO (T3.1b — Trực quan hóa)
# Vẽ biểu đồ kiểm tra hoặc đối chiếu tỉ trọng % (ví dụ: Cleveland dot plot hoặc cột nhóm/chồng 100% có ghi cỡ mẫu n).
# Lưu ý: Không sử dụng biểu đồ 3D hoặc hai trục tung (dual axis); biểu đồ cột bắt đầu từ 0.


### Kết quả & Nhận định

> **TODO (T3.1b):** Thành viên tự ghi nhận phát hiện dựa trên bằng chứng vừa chạy ở trên (không suy diễn vượt quá dữ liệu).

- **Bảng tổng hợp các chỉ số chuẩn hóa hợp lệ giữa các nhóm doanh nghiệp:** ...
- **Các lưu ý về cỡ mẫu `n` hoặc sự nhạy cảm của tỉ lệ phần trăm:** ...


## Tác vụ T3.2a — Phân tích & vẽ nhịp vận hành theo giờ / ca / ngày thường–cuối tuần giữa các ngành

### Mục tiêu
Đối chiếu nhịp độ hoạt động trong ngày (theo khung giờ / ca sáng–chiều) và giữa ngày thường với cuối tuần giữa các ngành nghề khác nhau bằng kỹ thuật *Small Multiples* (Schwabish, hướng dẫn 4).

### Câu hỏi khám phá
- Nhịp hoạt động theo giờ (hoặc ca sáng `06:00–13:59` vs ca chiều `14:00–21:59`) khác nhau thế nào giữa ngành ăn uống/cà phê, bán lẻ và dịch vụ?
- Mật độ giao dịch cuối tuần (Thứ 7 – Chủ nhật) tăng hay giảm so với ngày thường (Thứ 2 – Thứ 6) ở từng loại hình kinh doanh?
- Thiết kế biểu đồ *Small Multiples* dùng chung thang trục giúp lộ rõ sự khác biệt nhịp vận hành ra sao so với biểu đồ spaghetti?

### Các bước thực hiện

**Bước 1 — Lọc các doanh nghiệp đủ điều kiện về mốc giờ / ngày hoạt động**
Tách riêng nhóm doanh nghiệp có mốc giờ `HH:MM` (cho phân tích theo giờ/ca) và nhóm có đầy đủ ngày đặt hàng theo lịch (cho phân tích theo thứ trong tuần).

**Bước 2 — Tính phân bố tỷ trọng hoạt động theo giờ và theo ca (`rulebook.md`)**
Áp dụng đúng khung giờ `rulebook.md` (ca sáng `06:00–13:59`, ca chiều `14:00–21:59`; khung giờ tròn từ `h` đến `< h+1`) và chuẩn hóa về % trên tổng đơn của từng doanh nghiệp.

**Bước 3 — Tính mật độ hoạt động ngày thường vs. cuối tuần**
Sử dụng `calendar_df` (hoặc `weekday_counts_from_dates`) để chia tổng số đơn/ngày theo đúng số ngày Thứ 2..Thứ 6 và Thứ 7..Chủ nhật trong vùng phủ của mỗi doanh nghiệp.

**Bước 4 — Dựng biểu đồ Small Multiples và lưu vào `PHASE3_FIGURES_DIR`**
Dùng `FacetGrid` hoặc `plt.subplots(..., sharey=True)` để mỗi doanh nghiệp/ngành nằm trên một ô nhỏ cùng thang trục và lưu ảnh vào `PHASE3_FIGURES_DIR`.

**Bước 5 — Lưu bảng tóm tắt `operating_rhythm_summary.csv`**
Tổng hợp kết quả số liệu nhịp vận hành vào `PHASE3_TABLES_DIR / 'operating_rhythm_summary.csv'`.

### Gợi ý phương pháp & Lưu ý
- Tránh vẽ 6–10 đường chồng lên nhau trong một khung hình duy nhất (biểu đồ spaghetti) — hãy dùng **Small Multiples** cùng thang trục `sharey=True`.
- Nhắc lại quy ước `rulebook.md`: tuần bắt đầu từ Thứ Hai; trung bình mỗi ngày thứ X phải tính cả những ngày không có đơn trong khoảng thời gian doanh nghiệp đang hoạt động.

### Triển khai


In [ ]:
# TODO (T3.2a — Bước 1: Lọc các doanh nghiệp đủ điều kiện về mốc giờ / ngày hoạt động)
# TODO: Lọc danh sách doanh nghiệp đủ điều kiện cho nhịp giờ/ca và nhịp thứ trong tuần


In [ ]:
# TODO (T3.2a — Bước 2: Tính phân bố tỷ trọng hoạt động theo giờ và theo ca (`rulebook.md`))
# TODO: Tính tỷ trọng % số đơn (hoặc lượt) theo từng khung giờ và theo ca sáng/chiều


In [ ]:
# TODO (T3.2a — Bước 3: Tính mật độ hoạt động ngày thường vs. cuối tuần)
# TODO: Tính trung bình số đơn/ngày theo ngày thường vs cuối tuần và tỷ lệ chênh lệch %


In [ ]:
# TODO (T3.2a — Bước 4: Dựng biểu đồ Small Multiples và lưu vào `PHASE3_FIGURES_DIR`)
# TODO: Dựng biểu đồ Small Multiples cho nhịp giờ/ngày thường vs cuối tuần và lưu vào PHASE3_FIGURES_DIR


In [ ]:
# TODO (T3.2a — Bước 5: Lưu bảng tóm tắt `operating_rhythm_summary.csv`)
# TODO: Tóm tắt số liệu nhịp giờ/ca và cuối tuần vào operating_rhythm_summary.csv tại PHASE3_TABLES_DIR


### Trực quan hóa


In [ ]:
# TODO (T3.2a — Trực quan hóa)
# Vẽ biểu đồ Small Multiples (nhịp hoạt động theo giờ trong ngày hoặc mật độ ngày thường vs cuối tuần) dùng chung thang trục, có nhãn trực tiếp điểm cao điểm và lưu vào `PHASE3_FIGURES_DIR`.
# Lưu ý: Không sử dụng biểu đồ 3D hoặc hai trục tung (dual axis); biểu đồ cột bắt đầu từ 0.


### Kết quả & Nhận định

> **TODO (T3.2a):** Thành viên tự ghi nhận phát hiện dựa trên bằng chứng vừa chạy ở trên (không suy diễn vượt quá dữ liệu).

- **Sự khác biệt nhịp hoạt động theo giờ/ca giữa các ngành có mốc giờ:** ...
- **Mẫu hình ngày thường vs. cuối tuần quan sát được giữa các doanh nghiệp đủ độ phủ:** ...


## Tác vụ T3.2b — Phân tích & vẽ biến động dịp Tết / trước Tết qua `calendar.csv`

### Mục tiêu
Tra cứu lịch âm trong `calendar.csv` để đánh giá biến động hoạt động kinh doanh trước, trong và sau kỳ Tết (23 tháng Chạp đến mùng 10 tháng Giêng) chỉ trên những doanh nghiệp có dữ liệu bao phủ trọn vẹn cửa sổ này.

### Câu hỏi khám phá
- Theo `calendar.csv` và `rulebook.md`, kỳ Tết năm 2025 và 2026 tương ứng với những ngày dương lịch nào?
- Trong 14 doanh nghiệp, những doanh nghiệp nào thực sự có dữ liệu bao phủ đầy đủ giai đoạn *Trước Tết*, *Trong Tết* và *Sau Tết*?
- Hoạt động bán hàng/dịch vụ của các doanh nghiệp đủ độ phủ biến động ra sao trước Tết và trong kỳ Tết (đóng cửa nghỉ Tết vs. tăng đột biến trước Tết)?

### Các bước thực hiện

**Bước 1 — Xác định chính xác mốc dương lịch kỳ Tết và kỳ 'Trước Tết' từ `calendar.csv`**
Áp dụng quy tắc `rulebook.md`: kỳ Tết từ `23 tháng Chạp` đến hết `mùng 10 tháng Giêng`; kỳ *trước Tết* có cùng số ngày ngay liền trước 23 tháng Chạp.

**Bước 2 — Sàng lọc doanh nghiệp đủ độ phủ dữ liệu cho kỳ Tết**
Kiểm tra `[min_date, max_date]` và các khoảng trống của từng doanh nghiệp: loại bỏ những doanh nghiệp bắt đầu dữ liệu *sau* Tết hoặc bị thiếu hẳn giai đoạn so sánh.

**Bước 3 — Tính chỉ số biến động dịp Tết & lưu `tet_window_comparison.csv`**
Chuẩn hóa số đơn/doanh thu trung bình mỗi ngày trong các giai đoạn Tết và lưu kết quả vào `PHASE3_TABLES_DIR / 'tet_window_comparison.csv'`.

**Bước 4 — Trực quan hóa biến động dịp Tết & lưu vào `PHASE3_FIGURES_DIR`**
Vẽ biểu đồ chuỗi thời gian có vùng tô nền (`axvspan`) hoặc slope chart / diverging bar so sánh *Trước Tết* vs *Trong Tết* và lưu ảnh vào `PHASE3_FIGURES_DIR`.

**Bước 5 — Kiểm tra các yếu tố gây nhiễu (nghỉ Tết vs thiếu dữ liệu)**
Phân biệt rõ chuỗi ngày có 0 giao dịch trong kỳ Tết là do *cửa hàng nghỉ Tết* hay *thiếu nguồn dữ liệu*; không suy luận mùa vụ khi dữ liệu chỉ có 1 mùa Tết.

### Gợi ý phương pháp & Lưu ý
- Không tự tính lịch âm bằng tay — sử dụng trực tiếp cột `lunar_day`, `lunar_month`, `lunar_year`, `lunar_leap` trong `calendar.csv` hoặc `calendar_registry.tet_windows`.
- Nếu dữ liệu của một doanh nghiệp mới bắt đầu từ giữa năm 2025 hoặc kết thúc trước Tết 2026 thì **không được** đưa vào so sánh Tết năm đó.

### Triển khai


In [ ]:
# TODO (T3.2b — Bước 1: Xác định chính xác mốc dương lịch kỳ Tết và kỳ 'Trước Tết' từ `calendar.csv`)
# TODO: Từ calendar_df và calendar_registry.tet_windows, xác định rõ ngày bắt đầu/kết thúc của Trước Tết, Trong Tết, Sau Tết


In [ ]:
# TODO (T3.2b — Bước 2: Sàng lọc doanh nghiệp đủ độ phủ dữ liệu cho kỳ Tết)
# TODO: Lọc danh sách doanh nghiệp có phạm vi ngày bao trùm cả kỳ Trước Tết và Trong Tết


In [ ]:
# TODO (T3.2b — Bước 3: Tính chỉ số biến động dịp Tết & lưu `tet_window_comparison.csv`)
# TODO: Tính chỉ số trung bình ngày, % biến động qua các giai đoạn Tết và xuất CSV vào PHASE3_TABLES_DIR


In [ ]:
# TODO (T3.2b — Bước 4: Trực quan hóa biến động dịp Tết & lưu vào `PHASE3_FIGURES_DIR`)
# TODO: Vẽ biểu đồ biến động hoạt động trước/trong/sau kỳ Tết và lưu vào PHASE3_FIGURES_DIR


In [ ]:
# TODO (T3.2b — Bước 5: Kiểm tra các yếu tố gây nhiễu (nghỉ Tết vs thiếu dữ liệu))
# TODO: Kiểm tra nhật ký ngày giao dịch ngay trước và ngay sau kỳ nghỉ để xác nhận khoảng nghỉ Tết


### Trực quan hóa


In [ ]:
# TODO (T3.2b — Trực quan hóa)
# Vẽ biểu đồ đường theo ngày/tuần quanh cửa sổ Tết (hoặc biểu đồ so sánh % thay đổi Trước Tết vs Trong Tết) kèm đánh dấu `axvspan`, ghi rõ năm Tết quan sát và lưu vào `PHASE3_FIGURES_DIR`.
# Lưu ý: Không sử dụng biểu đồ 3D hoặc hai trục tung (dual axis); biểu đồ cột bắt đầu từ 0.


### Kết quả & Nhận định

> **TODO (T3.2b):** Thành viên tự ghi nhận phát hiện dựa trên bằng chứng vừa chạy ở trên (không suy diễn vượt quá dữ liệu).

- **Danh sách doanh nghiệp đủ độ phủ cho phân tích kỳ Tết (năm nào?):** ...
- **Các mẫu hình biến động quan sát được trước và trong dịp Tết theo từng nhóm ngành:** ...
- **Phân biệt khoảng nghỉ Tết thực tế với giới hạn độ phủ dữ liệu:** ...


## Tác vụ T3.2c — Tổng hợp giới hạn đối chiếu và trường hợp không thể so sánh trực tiếp

### Mục tiêu
Rà soát toàn bộ kết quả Hồi 3 để lập danh sách minh bạch các giới hạn về ngữ nghĩa, đơn vị đo, độ phủ thời gian và những trường hợp bị chặn không thể so sánh trực tiếp.

### Câu hỏi khám phá
- Những cặp hoặc chiều chỉ số nào đã bị loại khỏi phép so sánh liên doanh nghiệp và vì sao?
- Sự khác biệt về ngày cutoff (tháng 05–09/2026) và khoảng trống dữ liệu ảnh hưởng thế nào tới phạm vi tổng quát hóa của Hồi 3?
- Thông điệp cốt lõi cuối cùng của Hồi 3 (liên kết Hồi 1 và Hồi 2 thành câu chuyện hoàn chỉnh) là gì?

### Các bước thực hiện

**Bước 1 — Rà soát toàn bộ các phép đối chiếu đã thực hiện ở `T3.1a`–`T3.2b`**
Kiểm tra lại từng biểu đồ/bảng đối chiếu xem có vi phạm ranh giới grain, đơn vị hoặc cửa sổ thời gian nào không.

**Bước 2 — Lập bảng `limitations_df` và xuất `cross_business_limitations.csv`**
Tổng hợp thành bảng gồm: *Chiều so sánh*, *Các doanh nghiệp đủ điều kiện*, *Các doanh nghiệp loại trừ*, *Lý do kỹ thuật/nghiệp vụ* và lưu vào `PHASE3_TABLES_DIR / 'cross_business_limitations.csv'`.

**Bước 3 — Kiểm tra diễn đạt thận trọng & tính trung thực khoa học**
Đảm bảo mọi câu văn kết luận đều dùng diễn đạt có phạm vi rõ ràng (*'trong giai đoạn có dữ liệu của N doanh nghiệp đủ điều kiện'* thay vì khẳng định chung cho mọi ngành).

**Bước 4 — Đúc kết thông điệp chuyển tiếp từ Hồi 3 sang Kết luận / Hành động (Giai đoạn 4)**
Tổng hợp mạch kết nối 3 hồi: *Bối cảnh (14 DN) -> Bằng chứng vi mô (từng DN đại diện) -> Bằng chứng đối chiếu (liên ngành) -> Kết luận & Khuyến nghị*.

### Gợi ý phương pháp & Lưu ý
- Mục này tập trung vào tổng hợp, phản biện và tính trung thực dữ liệu — không cần vẽ biểu đồ mới trừ khi muốn minh họa tóm tắt phạm vi loại trừ.
- Các ghi chú giới hạn lập ở đây (`cross_business_limitations.csv`) sẽ được kiểm tra kèm trong Notebook 04.

### Triển khai


In [ ]:
# TODO (T3.2c — Bước 1: Rà soát toàn bộ các phép đối chiếu đã thực hiện ở `T3.1a`–`T3.2b`)
# TODO: Rà soát kiểm toán lại các phép đối chiếu đã chạy ở T3.1b, T3.2a, T3.2b


In [ ]:
# TODO (T3.2c — Bước 2: Lập bảng `limitations_df` và xuất `cross_business_limitations.csv`)
# TODO: Tạo DataFrame limitations_df và xuất ra PHASE3_TABLES_DIR / 'cross_business_limitations.csv'
limitations_df = pd.DataFrame()


In [ ]:
# TODO (T3.2c — Bước 3: Kiểm tra diễn đạt thận trọng & tính trung thực khoa học)
# TODO: Soạn danh sách các ghi chú giới hạn ngắn gọn để gắn lên caption biểu đồ / slide


In [ ]:
# TODO (T3.2c — Bước 4: Đúc kết thông điệp chuyển tiếp từ Hồi 3 sang Kết luận / Hành động (Giai đoạn 4))
# TODO: Hoàn thiện cấu trúc thông điệp bàn giao sang Notebook 04


### Kết quả & Nhận định

> **TODO (T3.2c):** Thành viên tự ghi nhận phát hiện dựa trên bằng chứng vừa chạy ở trên (không suy diễn vượt quá dữ liệu).

- **Bảng tóm tắt các trường hợp không thể so sánh trực tiếp và lý do:** ...
- **Thông điệp chính và ghi chú giới hạn bắt buộc của Hồi 3 bàn giao cho Giai đoạn 4:** ...


## 05. Tổng kết giai đoạn & Bàn giao

> **TODO (Thành viên phụ trách Giai đoạn 3):** Hoàn thiện checklist trước khi bàn giao sang Giai đoạn 4.

- **Những công việc đã thực hiện:**
  - [ ] `T3.1a`: Lập ma trận khả năng so sánh -> `tables/comparability_matrix.csv`.
  - [ ] `T3.1b`: Chuẩn hóa chỉ số về tỉ lệ/phần trăm -> `tables/normalized_cross_metrics.csv`.
  - [ ] `T3.2a`: Phân tích & dựng Small Multiples nhịp vận hành -> `tables/operating_rhythm_summary.csv`, `figures/`.
  - [ ] `T3.2b`: Phân tích biến động dịp Tết -> `tables/tet_window_comparison.csv`, `figures/`.
  - [ ] `T3.2c`: Tổng hợp giới hạn đối chiếu -> `tables/cross_business_limitations.csv`.
- **Những phát hiện chính của Hồi 3:**
  - ...
- **Các vấn đề còn tồn tại / Các phép so sánh bị giới hạn:**
  - ...
- **Giới hạn của kết quả:**
  - ...
- **Đầu ra bàn giao cho Giai đoạn 4 (`04_visualization_and_storytelling.ipynb`):**
  - `artifacts/guided_eda/phase3_cross_business/tables/`
  - `artifacts/guided_eda/phase3_cross_business/figures/`
